### Step-0: Testing Connection

In [0]:
%pip install confluent-kafka

In [0]:
%pip install confluent-kafka[avro]
dbutils.library.restartPython()

In [0]:
import importlib, schema_registry_config
importlib.reload(schema_registry_config)
from schema_registry_config import *
reg_conf = get_kafka_registry_config(dbutils)

In [0]:
reg_conf

In [0]:
from confluent_kafka.schema_registry import SchemaRegistryClient

sr_client = SchemaRegistryClient(reg_conf)

# this call hits the registry — proves connectivity + auth
subjects = sr_client.get_subjects()
print("Connected to Schema Registry.")
print("Existing subjects:", subjects)

### Create a Topic

In [0]:
import importlib, kafka_conn
importlib.reload(kafka_conn)
from kafka_conn import *

TOPIC = kafka_conn.TOPIC_REGISTRY
conf = kafka_conn.get_kafka_conf(dbutils)

In [0]:
from confluent_kafka.admin import AdminClient, NewTopic

admin = AdminClient({
    "bootstrap.servers": conf["bootstrap.servers"],
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": conf["sasl.username"],
    "sasl.password": conf["sasl.password"],
})

futures = admin.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=3)])
for name, f in futures.items():
    try:
        f.result(); print(f"Created topic: {name}")
    except Exception as e:
        print(f"{name}: {e}")   # 'already exists' is fine on re-run
        

### Schema Register